# Day 3: Dynamic Pages, Selenium, Debugging, and Robust Scraping

Some websites do not put the visible data into the first HTML response. They load content later with JavaScript, after the browser has opened the page.

This notebook starts with a short Wikipedia comparison, because Day 1 used the MediaWiki API and Day 2 used static Wikipedia HTML scraping. Then it moves to a JavaScript-rendered page where browser automation is actually needed.

By the end, you should understand:

- how browser automation differs from API access and static scraping
- why `requests` may miss JavaScript-rendered content
- how Selenium observes a rendered browser DOM
- how wait conditions shape collected data
- how scrolling and infinite-scroll stopping rules work
- why screenshots and rendered HTML are raw evidence
- how to debug empty or broken scrapers
- what bot-compliance boundaries should not be crossed


## Packages Used in This Notebook

This notebook uses:

- `requests` to fetch the initial server HTML without JavaScript
- Selenium inside functions to control a real browser
- `pandas` to turn extracted records into a table
- `Path` to save rendered HTML, screenshots, CSVs, and diagnostics
- `time` to pause after browser actions such as scrolling

The core contrast is:

```text
requests -> initial HTML from the server
Selenium  -> rendered DOM after the browser runs JavaScript
```


In [2]:
# time lets us pause after browser actions such as scrolling.
import time
# Path helps create output folders and file paths.
from pathlib import Path

# pandas turns extracted browser records into tables.
import pandas as pd
# requests fetches the initial server HTML without running JavaScript.
import requests


## Same Source, Third Access Route: Wikipedia in a Browser

First we open the same Wikipedia page in an automated browser.

This is useful for comparison, not because Wikipedia requires Selenium for basic article text. The API and static HTML route are usually better for this source. Here, Selenium helps us see what browser automation adds:

- rendered page state
- browser title and URL
- screenshots
- waits for visible page elements
- rendered HTML as browser evidence


In [3]:
USER_AGENT = "methodsNET-VLOP-course/1.0 dynamic walkthrough"
WIKIPEDIA_URL = "https://en.wikipedia.org/wiki/Digital_Services_Act"

outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
reports_dir = outdir / "reports"

raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)


## 1. Browser Automation Where It Is Possible but Not Necessary

This function opens Wikipedia in Chrome, waits for the article heading, extracts a few simple fields, and saves a screenshot.

The point is to compare access routes:

- API: structured JSON from MediaWiki
- static scraping: HTML from `requests`
- browser automation: rendered DOM and screenshot from a real browser

For this particular page, browser automation is overkill. That is an important methodological conclusion.


In [ ]:
def open_wikipedia_with_selenium(url, *, headless=True):
    """Open a Wikipedia page in Chrome and collect rendered-page evidence."""

    # Selenium imports live inside the function so the notebook can still be read
    # before browser dependencies are installed.
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    options = Options()
    options.add_argument(f"--user-agent={USER_AGENT}")

    # headless=True runs without a visible window; set False for a live demo.
    if headless:
        options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)

    try:
        # Open the page in a real browser.
        driver.get(url)

        # Wait until the article heading exists in the rendered DOM.
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))

        # Extract a few simple browser-observed fields.
        title = driver.title
        heading = driver.find_element(By.CSS_SELECTOR, "h1").text
        paragraph_count = len(driver.find_elements(By.CSS_SELECTOR, ".mw-parser-output p"))
        link_count = len(driver.find_elements(By.CSS_SELECTOR, ".mw-parser-output a[href]"))

        # page_source is the rendered DOM as seen by the browser.
        rendered_html = driver.page_source
        # A screenshot is visual evidence of the rendered page state.
        screenshot = driver.get_screenshot_as_png()

    finally:
        # Always close the browser, even if extraction fails.
        driver.quit()

    return {
        "url": url,
        "browser_title": title,
        "heading": heading,
        "paragraph_count": paragraph_count,
        "link_count": link_count,
        "rendered_html": rendered_html,
        "screenshot": screenshot,
    }


wiki_browser = open_wikipedia_with_selenium(WIKIPEDIA_URL, headless=True)

print("Browser title:", wiki_browser["browser_title"])
print("Heading:", wiki_browser["heading"])
print("Paragraph count:", wiki_browser["paragraph_count"])
print("Link count:", wiki_browser["link_count"])


## 2. Save Wikipedia Browser Evidence

For browser automation, raw evidence should include not only extracted tables but also rendered HTML and screenshots. These help diagnose cookie banners, broken rendering, empty states, or unexpected page versions.


In [ ]:
wiki_rendered_path = raw_dir / "notebook_wikipedia_dsa_selenium_rendered.html"
wiki_screenshot_path = raw_dir / "notebook_wikipedia_dsa_selenium.png"
wiki_diagnostics_path = reports_dir / "notebook_wikipedia_dsa_selenium_diagnostics.json"

wiki_rendered_path.write_text(wiki_browser["rendered_html"], encoding="utf-8")
wiki_screenshot_path.write_bytes(wiki_browser["screenshot"])

pd.Series(
    {
        "url": wiki_browser["url"],
        "browser_title": wiki_browser["browser_title"],
        "heading": wiki_browser["heading"],
        "paragraph_count": wiki_browser["paragraph_count"],
        "link_count": wiki_browser["link_count"],
        "method_note": "Browser automation works here, but API/static scraping are usually better for this Wikipedia task.",
    }
).to_json(wiki_diagnostics_path, indent=2)

print(wiki_rendered_path)
print(wiki_screenshot_path)
print(wiki_diagnostics_path)


## Main Dynamic Example: Quotes to Scrape JS

Now we switch to a page where browser automation is more clearly justified.

`quotes.toscrape.com/js/` is a teaching page where the visible quotes are rendered by JavaScript. That makes it useful for comparing what `requests` sees with what a browser sees.


In [ ]:
URL = "https://quotes.toscrape.com/js/"


## 3. Fetch the JS Page with `requests` First

Always start by checking the simple route.

`requests.get()` asks the server for the HTML document, but it does not execute JavaScript, click buttons, scroll, or wait for client-side rendering.


In [4]:
# requests.get() sends a normal HTTP GET request.
static_response = requests.get(URL, headers={"User-Agent": USER_AGENT}, timeout=30)

# raise_for_status() turns HTTP error statuses into visible Python errors.
static_response.raise_for_status()

# response.text is the initial HTML returned by the server.
static_html = static_response.text

print("Static HTML characters:", len(static_html))
print("Does static HTML contain quote text?", "The world as we have created it" in static_html)


Static HTML characters: 5806
Does static HTML contain quote text? True


## 4. Diagnose: Rendering Problem or Access Problem?

A missing record can mean different things.

It might be a rendering problem:

- the server returned HTML
- the browser later adds content with JavaScript
- Selenium may be appropriate

Or it might be an access/compliance problem:

- `403 Forbidden`
- `429 Too Many Requests`
- login wall
- paywall
- CAPTCHA / prove-you-are-human page
- consent wall that changes access conditions

Browser automation is for observing rendered public content. It should not be used to bypass access controls.


In [5]:
diagnosis = {
    "status_code": static_response.status_code,
    "content_type": static_response.headers.get("content-type"),
    "static_html_characters": len(static_html),
    "contains_expected_quote_text": "The world as we have created it" in static_html,
    "contains_captcha_word": "captcha" in static_html.lower(),
    "contains_login_word": "login" in static_html.lower(),
}

diagnosis


{'status_code': 200,
 'content_type': 'text/html; charset=utf-8',
 'static_html_characters': 5806,
 'contains_expected_quote_text': True,
 'contains_captcha_word': False,
 'contains_login_word': True}

## 5. What Selenium Is For

Selenium is a browser-automation tool. It lets Python control a real browser in roughly the way a user would:

- open a URL;
- wait until an element appears;
- read visible text;
- read attributes such as `href`, `src`, or `aria-label`;
- click buttons or links;
- type into forms;
- submit a form;
- scroll;
- take screenshots;
- save the rendered HTML.

Selenium is useful when the simple static route is insufficient, for example when:

- content is added by JavaScript after the first HTML response;
- a page requires clicking a tab, button, menu, or “load more” control to reveal public content;
- the page uses infinite scroll;
- you need a screenshot or rendered-page evidence;
- you need to test whether selectors work in the browser DOM, not only in saved HTML.

Selenium is usually **not** the best first choice when an API or simple static HTML request gives the needed data more cleanly. It is slower, more fragile, and easier to misuse.


## 6. Common Selenium Operations

The examples below are reference patterns. They are not all needed for the quotes example, but they are important to recognize when reading or adapting browser-automation scripts.


In [ ]:
# This cell is a reference sheet, not a workflow to run as-is.
# It assumes a Selenium driver already exists:
# driver = webdriver.Chrome(options=options)

# Open a page.
# driver.get("https://example.com")

# Find one element by CSS selector.
# heading = driver.find_element("css selector", "h1")
# print(heading.text)

# Find many elements by CSS selector.
# links = driver.find_elements("css selector", "a[href]")
# for link in links:
#     print(link.text, link.get_attribute("href"))

# Wait until an element exists before trying to use it.
# wait = WebDriverWait(driver, 15)
# wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

# Click a button or link.
# button = driver.find_element("css selector", "button.load-more")
# button.click()

# Type into a search box or form field.
# search_box = driver.find_element("css selector", "input[name='q']")
# search_box.clear()
# search_box.send_keys("digital services act")

# Submit by pressing Enter.
# from selenium.webdriver.common.keys import Keys
# search_box.send_keys(Keys.ENTER)

# Take a screenshot.
# driver.get_screenshot_as_file("page_state.png")

# Save rendered HTML.
# rendered_html = driver.page_source


## 7. Login and Form Automation: Important Boundary

Selenium can type into username and password fields, but this should be treated carefully.

Acceptable teaching/research cases might include:

- logging into your own account when the platform permits automated access;
- using a test account in a controlled teaching environment;
- automating an internal tool with permission;
- documenting a workflow where the credentials and access rights are already legitimate.

Do **not** use Selenium to bypass paywalls, CAPTCHAs, “prove you are human” checks, account restrictions, rate limits, or platform access controls.

Never hard-code passwords in a script. If credentials are needed, read them from environment variables or enter them interactively.


In [ ]:
# Example pattern for a permitted/test login form.
# Do not run this against real platforms unless you have permission and the site's rules allow it.

# import os
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.webdriver.support.ui import WebDriverWait

# username = os.getenv("COURSE_DEMO_USERNAME")
# password = os.getenv("COURSE_DEMO_PASSWORD")

# if username is None or password is None:
#     raise RuntimeError("Set COURSE_DEMO_USERNAME and COURSE_DEMO_PASSWORD first.")

# driver.get("https://example.com/login")
# wait = WebDriverWait(driver, 15)

# username_box = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='username']")))
# password_box = driver.find_element(By.CSS_SELECTOR, "input[name='password']")

# username_box.clear()
# username_box.send_keys(username)
# password_box.clear()
# password_box.send_keys(password)

# submit_button = driver.find_element(By.CSS_SELECTOR, "button[type='submit']")
# submit_button.click()

# Wait for evidence that login succeeded, such as a known page element.
# wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".account-dashboard")))


## 8. Browser Helper Functions

Selenium controls a real browser from Python.

We define helpers for three reasons:

1. keep the browser workflow readable
2. make diagnostics reproducible
3. make scrolling use an explicit stopping rule


In [ ]:
def describe_browser_state(driver):
    """Return small diagnostics about what the browser currently sees."""

    return {
        "current_url": driver.current_url,
        "title": driver.title,
        "html_characters": len(driver.page_source),
        "quote_count": len(driver.find_elements("css selector", ".quote")),
        "link_count": len(driver.find_elements("css selector", "a[href]")),
    }


def scroll_until_stable(driver, *, record_selector, max_scrolls=5, wait_seconds=1.0):
    """Scroll until record count stops changing or max_scrolls is reached."""

    counts = []
    previous_count = len(driver.find_elements("css selector", record_selector))
    counts.append(previous_count)

    for scroll_number in range(1, max_scrolls + 1):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(wait_seconds)

        current_count = len(driver.find_elements("css selector", record_selector))
        counts.append(current_count)

        print(f"Scroll {scroll_number}: {current_count} records")

        # Stop when scrolling no longer reveals additional records.
        if current_count == previous_count:
            break
        previous_count = current_count

    return counts


## 9. Selenium Collection Function

This function opens a browser, waits for rendered quote blocks, scrolls with a stopping rule, extracts records, and takes a screenshot.

Important methodological choices:

- `headless`: whether the browser is visible
- wait selector: here `.quote`
- scroll stopping rule: stop when record count stops increasing
- raw evidence: rendered HTML and screenshot


In [ ]:
def collect_rendered_page_with_selenium(url, *, headless=True, max_scrolls=2, wait_seconds=1.0):
    """Open a page in Chrome, wait for rendered content, and extract quote rows."""

    # Selenium imports live inside the function so the notebook can be read even
    # before Selenium/browser dependencies are installed.
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait

    options = Options()
    options.add_argument(f"--user-agent={USER_AGENT}")

    # headless=True runs without a visible browser window.
    # During live teaching, set headless=False so students can watch the browser.
    if headless:
        options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)

    try:
        # driver.get() opens the page in the automated browser.
        driver.get(url)

        # Wait until at least one rendered quote block exists.
        # This is stronger than waiting for the body alone.
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

        before_scroll = describe_browser_state(driver)

        # Scroll with a record-count stopping rule.
        scroll_counts = scroll_until_stable(
            driver,
            record_selector=".quote",
            max_scrolls=max_scrolls,
            wait_seconds=wait_seconds,
        )

        after_scroll = describe_browser_state(driver)

        # page_source is rendered DOM after JavaScript execution.
        rendered_html = driver.page_source

        quote_blocks = driver.find_elements(By.CSS_SELECTOR, ".quote")
        quotes = []

        for block in quote_blocks:
            quote_text = block.find_element(By.CSS_SELECTOR, ".text").text
            author = block.find_element(By.CSS_SELECTOR, ".author").text
            tags = [tag.text for tag in block.find_elements(By.CSS_SELECTOR, ".tag")]

            quotes.append({"quote": quote_text, "author": author, "tags": tags})

        # Screenshot is visual audit evidence.
        screenshot = driver.get_screenshot_as_png()

    finally:
        # Always close automated browsers.
        driver.quit()

    diagnostics = {
        "before_scroll": before_scroll,
        "after_scroll": after_scroll,
        "scroll_counts": scroll_counts,
    }

    return rendered_html, quotes, screenshot, diagnostics


## 10. Run the Browser Collection

For class, consider setting `headless=False` once so students can see the browser open and render.


In [ ]:
rendered_html, quotes, screenshot, diagnostics = collect_rendered_page_with_selenium(
    URL,
    headless=True,
    max_scrolls=2,
    wait_seconds=1.0,
)

print("Rendered HTML characters:", len(rendered_html))
print("Quotes extracted:", len(quotes))
print("Diagnostics:", diagnostics)

pd.DataFrame(quotes).head()


## 11. Save JS-Page Evidence

For dynamic pages, a CSV alone is not enough.

Save:

- rendered HTML: what the browser saw after JavaScript
- screenshot: visual audit of page state
- processed CSV: extracted table
- diagnostics JSON: counts, scroll history, and other checks


In [ ]:
rendered_path = raw_dir / "notebook_rendered_quotes_selenium.html"
screenshot_path = raw_dir / "notebook_rendered_quotes_selenium.png"
quotes_path = processed_dir / "notebook_rendered_quotes_selenium.csv"
diagnostics_path = reports_dir / "notebook_rendered_quotes_selenium_diagnostics.json"

rendered_path.write_text(rendered_html, encoding="utf-8")
screenshot_path.write_bytes(screenshot)
pd.DataFrame(quotes).to_csv(quotes_path, index=False)
pd.Series(diagnostics).to_json(diagnostics_path, indent=2)

print(rendered_path)
print(screenshot_path)
print(quotes_path)
print(diagnostics_path)


## 12. Debugging Checklist

When a scraper returns zero records or strange records, debug in layers.


In [ ]:
debugging_checklist = [
    "Check response.status_code before parsing anything.",
    "Compare response.text with what the browser visibly shows.",
    "Save raw static HTML and rendered HTML for comparison.",
    "Inspect one record container before looping over all records.",
    "Use explicit waits for content-specific selectors, not only fixed sleep times.",
    "Save a screenshot to catch cookie banners, empty states, or blocked pages.",
    "Count records before and after scrolling; define a stopping rule.",
    "If selectors return zero, inspect whether markup changed.",
    "If you see login, paywall, CAPTCHA, or prove-you-are-human pages, stop and document.",
]

for item in debugging_checklist:
    print("-", item)


## 13. Anti-Fragile Patterns and Anti-Patterns

Anti-fragile does not mean impossible to break. It means the scraper is designed so breakage is visible, documented, and easier to repair.


In [ ]:
patterns = {
    "good_patterns": [
        "Wait for a content-specific selector.",
        "Save raw/rendered evidence and screenshots.",
        "Use small test runs before scaling.",
        "Keep selectors close to the record container.",
        "Log counts, URLs, status codes, and output paths.",
        "Stop when scroll counts no longer increase.",
        "Prefer official APIs or data downloads when available.",
    ],
    "anti_patterns": [
        "Using time.sleep() as the only waiting strategy.",
        "Looping forever on infinite scroll without a stopping rule.",
        "Ignoring empty outputs because the script did not crash.",
        "Treating a CAPTCHA or login wall as a technical obstacle to bypass.",
        "Collecting rapidly without rate limits or documentation.",
        "Saving only the final CSV and no raw evidence.",
    ],
}

for label, items in patterns.items():
    print(label)
    for item in items:
        print("-", item)


## 14. Bot-Compliance Decision Tree

This course teaches browser automation as a way to observe rendered public pages, not as a way to evade restrictions.


In [ ]:
bot_compliance_decision_tree = [
    "If content is missing because JavaScript renders it: browser automation may be appropriate.",
    "If a public API or bulk download exists: prefer that route when it fits the research question.",
    "If the site returns 429: pause and respect retry/rate-limit information.",
    "If the site returns 403: do not force access; check documentation and permissions.",
    "If the site shows CAPTCHA or human verification: stop; do not automate around it.",
    "If login, paywall, or terms restrict access: request permission or redesign the data source.",
    "Always document blocks, exclusions, and access limitations in provenance.",
]

for item in bot_compliance_decision_tree:
    print("-", item)


## 15. Where Playwright Fits

Selenium is a good starting point because it makes the idea of controlling a browser concrete.

Playwright is a newer browser-automation library with strong auto-waiting, browser contexts, network inspection, and tracing. For more complex modern web apps, Playwright can be easier to debug.

In this course:

- start with Selenium to understand browser automation
- mention Playwright as a modern alternative
- use the runnable Playwright script if you want to compare tools or show network-oriented debugging


## Exercise

Choose one debugging scenario and write what you would check:

1. `requests` returns HTML, but the target records are missing.
2. Selenium opens the page, but extracts zero records.
3. Scrolling does not increase the record count.
4. The screenshot shows a cookie/consent banner covering the page.
5. The page shows a CAPTCHA or login wall.

For each scenario, answer:

- Is this a rendering issue, selector issue, timing issue, or access/compliance boundary?
- What evidence would you save?
- What would you try next?
- What would you not do?

## Key Takeaway

Dynamic scraping is not just Selenium. It is diagnosis, waiting, evidence, stopping rules, debugging, and compliance.
